In [1]:
import os
import urllib.request
import gzip
import shutil

# 1. Download official fastText Burmese model (.bin.gz)
url = "https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.my.300.bin.gz"
gz_path = "cc.my.300.bin.gz"
bin_path = "cc.my.300.bin"

if not os.path.exists(bin_path):
    print("Downloading fastText Burmese model (approx. 4.2 GB compressed)...")
    urllib.request.urlretrieve(url, gz_path)
    
    print("Decompressing...")
    with gzip.open(gz_path, 'rb') as f_in:
        with open(bin_path, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)
    
    # Clean up compressed file
    os.remove(gz_path)
    print("Done! File saved as:", bin_path)
else:
    print("File already exists!")

Decompressing...
Done! File saved as: cc.my.300.bin


In [6]:
import os
import shutil
import kagglehub

# 1. Define paths
file_to_upload = "/kaggle/working/cc.my.300.bin"
temp_upload_dir = "/kaggle/working/temp_upload"
handle = "phyohtetpwint/fast-text"

# 2. Setup isolated folder with ONLY the target file
if os.path.exists(temp_upload_dir):
    shutil.rmtree(temp_upload_dir)
os.makedirs(temp_upload_dir, exist_ok=True)

shutil.copy(file_to_upload, os.path.join(temp_upload_dir, "cc.my.300.bin"))

# 3. Upload to Kaggle using correct argument syntax
kagglehub.dataset_upload(
    handle=handle,
    local_dataset_dir=temp_upload_dir,
    version_notes="Adding cc.my.300.bin"
)

# 4. Clean up temporary directory
shutil.rmtree(temp_upload_dir)

print("Upload complete!")

Uploading Dataset https://api.kaggle.com/datasets/phyohtetpwint/fast-text ...
Starting upload for file /kaggle/working/temp_upload/cc.my.300.bin


Uploading: 100%|██████████| 3.23G/3.23G [00:38<00:00, 84.8MB/s]

Upload successful: /kaggle/working/temp_upload/cc.my.300.bin (3GB)


Your dataset has been created.
Files are being processed...
See at: https://api.kaggle.com/datasets/phyohtetpwint/fast-text
Upload complete!


In [1]:
!pip install scikit-learn Gensim

#  Comparative Evaluation Report: POS Tagging and NER Sequence Labeling on the myNER Dataset using Support Vector Machine (SVM)

<span style="font-size: 20px;">*This report evaluates four distinct LinearSVC experimental configurations implemented on the Burmese [myNER](https://github.com/ye-kyaw-thu/myNER) dataset:*</span>

<span style="font-size: 18px;">***Model 1: Single-Task NER (Sparse Features + Gold POS Inputs) — Standard sequence classifier using sparse contextual features and ground-truth POS inputs.***</span>

In [1]:
# LinearSVC SVM model for NER (Sparse Features Only - No FastText)
import numpy as np
import time
import sys
import platform
import sklearn  
from sklearn.feature_extraction import DictVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, f1_score

def check_environment():
    print("=== Environment Details ===")
    print(f"Python Version: {sys.version}")
    print(f"Platform: {platform.system()} {platform.release()}")
    print(f"scikit-learn Version: {sklearn.__version__}")
    print("===========================")

# Check environment
check_environment()

# 1. Feature Extraction Function per Token (Sparse Features)
def is_numeric(word):
    numeric_chars = set("၁၂၃၄၅၆၇၈၉၀")
    return word.isdigit() or all(char in numeric_chars for char in word)
    
def token_to_features(sent, pos_tags, i):
    word = sent[i]
    pos = pos_tags[i]
    
    features = {
        'bias': 1.0,
        'word': word,
        'pos': pos,
        'prefix_2': word[:2],
        'prefix_3': word[:3],
        'suffix_2': word[-2:],
        'suffix_3': word[-3:],
        'has_hyphen': '-' in word,
        'is_numeric': is_numeric(word),
        'is_latin': word.isascii(),
    }
    
    # Context Windows
    if i > 0:
        features['w-1'] = sent[i-1]
        features['pos-1'] = pos_tags[i-1]
    else:
        features['BOS'] = True

    if i > 1:
        features['w-2'] = sent[i-2]
        features['pos-2'] = pos_tags[i-2]

    if i < len(sent) - 1:
        features['w+1'] = sent[i+1]
        features['pos+1'] = pos_tags[i+1]
    else:
        features['EOS'] = True

    if i < len(sent) - 2:
        features['w+2'] = sent[i+2]
        features['pos+2'] = pos_tags[i+2]
        
    return features

# 2. Process CoNLL Data
def prepare_dataset(file_path):
    sparse_features, labels = [], []
    raw_sentences = []
    
    with open(file_path, "r", encoding="utf-8") as f:
        sentence, pos_tags, ner_tags = [], [], []
        for line in f:
            stripped = line.strip()
            if stripped:
                parts = stripped.split("\t")
                if len(parts) == 3:
                    w, p, n = parts
                    # Strip spaces AND literal quote characters
                    w_clean = w.strip().strip("'\"")
                    
                    if w_clean:
                        sentence.append(w_clean)
                        pos_tags.append(p.strip())
                        ner_tags.append(n.strip())
            elif sentence:
                raw_sentences.append(list(zip(sentence, pos_tags, ner_tags)))
                for i in range(len(sentence)):
                    sparse_features.append(token_to_features(sentence, pos_tags, i))
                    labels.append(ner_tags[i])
                sentence, pos_tags, ner_tags = [], [], []
                
        if sentence:
            raw_sentences.append(list(zip(sentence, pos_tags, ner_tags)))
            for i in range(len(sentence)):
                sparse_features.append(token_to_features(sentence, pos_tags, i))
                labels.append(ner_tags[i])

    return sparse_features, labels, raw_sentences
    
# Load Datasets
train_sparse, train_y, train_sents = prepare_dataset("/kaggle/input/datasets/phyohtetpwint/myner-corpus/7-tags/corpus_ver.1.0/train_v5.conll")
test_sparse, test_y, test_sents = prepare_dataset("/kaggle/input/datasets/phyohtetpwint/myner-corpus/7-tags/corpus_ver.1.0/test_v5.conll")
val_sparse, val_y, val_sents = prepare_dataset("/kaggle/input/datasets/phyohtetpwint/myner-corpus/7-tags/corpus_ver.1.0/val_v5.conll")

# 3. Vectorize Sparse Features
vec = DictVectorizer(sparse=True)
train_X = vec.fit_transform(train_sparse)
test_X = vec.transform(test_sparse)
val_X = vec.transform(val_sparse)

# 4. Model Training
start_time = time.time()
print("Training LinearSVC model for NER (Sparse Features Only)...")
clf = LinearSVC(C=1.0, max_iter=2000, random_state=42)
clf.fit(train_X, train_y)
print(f"Training completed in {time.time() - start_time:.2f} seconds.")

# Model Validation
val_preds = clf.predict(val_X)
val_f1 = f1_score(val_y, val_preds, average='weighted')
print(f"NER Validation Weighted F1-score: {val_f1:.4f}")

# 5. Evaluation
preds = clf.predict(test_X)
print("\nLinearSVC SVM model (Sparse Features Only) for NER tags Classification Report:")
print(classification_report(test_y, preds, digits=4, zero_division=0))

# 6. Print 5 Predicted Sentences from Test Set
print("\n=== 5 Predicted Sentences from Test Set ===")
pred_idx = 0
for i in range(min(5, len(test_sents))):
    sentence_tokens = test_sents[i]
    print(f"\nSentence {i + 1}:")
    print(f"{'Token':<18} {'POS Tag':<12} {'True NER':<12} {'Predicted NER':<12}")
    print("-" * 56)
    for word, pos_tag, true_ner in sentence_tokens:
        pred_ner = preds[pred_idx]
        print(f"{word:<18} {pos_tag:<12} {true_ner:<12} {pred_ner:<12}")
        pred_idx += 1

=== Environment Details ===
Python Version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux 6.12.90+
scikit-learn Version: 1.6.1
Training LinearSVC model for NER (Sparse Features Only)...
Training completed in 129.16 seconds.
NER Validation Weighted F1-score: 0.9802

LinearSVC SVM model (Sparse Features Only) for NER tags Classification Report:
              precision    recall  f1-score   support

      B-DATE     0.8154    0.8030    0.8092        66
       B-LOC     0.9821    0.9721    0.9770      1182
       B-NUM     0.5000    0.4000    0.4444        15
       B-ORG     0.7500    0.5625    0.6429        48
       B-PER     0.8857    0.9118    0.8986        34
      B-TIME     0.8750    0.7778    0.8235         9
      E-DATE     0.8793    0.7727    0.8226        66
       E-LOC     0.9763    0.9746    0.9754      1182
       E-NUM     0.4167    0.3333    0.3704        15
       E-ORG     0.7317    0.6250    0.6742        48
       E-PER     0.8649    0.9412    0.

<span style="font-size: 18px;">***Model 2: Single-Task NER (Sparse + FastText Embeddings + Gold POS Inputs) — Extends Model 1 by concatenating continuous 300-dimensional FastText subword vectors (cc.my.300.bin).***</span>

In [1]:
#  LinearSVC SVM model with POS tags and fasttext embeddings
import numpy as np
import time
import sys
import platform
from scipy.sparse import hstack
import sklearn  
import gensim   
from sklearn.feature_extraction import DictVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, f1_score
from gensim.models.fasttext import load_facebook_model
import logging

def check_environment():
    print("=== Environment Details ===")
    print(f"Python Version: {sys.version}")
    print(f"Platform: {platform.system()} {platform.release()}")
    print(f"scikit-learn Version: {sklearn.__version__}")
    print(f"Gensim Version: {gensim.__version__}")
    print("===========================")

# Check environment
check_environment()

# 1. Load Pre-trained FastText Model
fasttext_model = load_facebook_model("/kaggle/input/datasets/phyohtetpwint/fast-text/cc.my.300.bin")
ft_wv = fasttext_model.wv

# 2. Feature Extraction Function per Token
def is_numeric(word):
    numeric_chars = set("၁၂၃၄၅၆၇၈၉၀")
    return word.isdigit() or all(char in numeric_chars for char in word)
    
def token_to_features(sent, pos_tags, i):
    word = sent[i]
    pos = pos_tags[i]
    
    features = {
        'bias': 1.0,
        'word': word,
        'pos': pos,
        'prefix_2': word[:2],
        'prefix_3': word[:3],
        'suffix_2': word[-2:],
        'suffix_3': word[-3:],
        'has_hyphen': '-' in word,
        'is_numeric': is_numeric(word),
        'is_latin': word.isascii(),
    }
    
    # Context Windows
    if i > 0:
        features['w-1'] = sent[i-1]
        features['pos-1'] = pos_tags[i-1]
    else:
        features['BOS'] = True

    if i > 1:
        features['w-2'] = sent[i-2]
        features['pos-2'] = pos_tags[i-2]

    if i < len(sent) - 1:
        features['w+1'] = sent[i+1]
        features['pos+1'] = pos_tags[i+1]
    else:
        features['EOS'] = True

    if i < len(sent) - 2:
        features['w+2'] = sent[i+2]
        features['pos+2'] = pos_tags[i+2]
        
    return features

# Suppress Gensim's verbose n-gram warnings globally
logging.getLogger("gensim.models.fasttext").setLevel(logging.ERROR)

def get_dense_embeddings(sent, i):
    word = sent[i].strip("'\" \t\n\r")  # Strip quotes and spaces
    
    # Return zero vector for empty strings or 0-length tokens after stripping
    if not word:
        return np.zeros(ft_wv.vector_size)
        
    try:
        return ft_wv[sent[i]]
    except Exception:
        return np.zeros(ft_wv.vector_size)

# 4. Process CoNLL Data (Preserving Sentence Structure)
def prepare_dataset(file_path):
    sparse_features, dense_features, labels = [], [], []
    raw_sentences = []
    
    with open(file_path, "r", encoding="utf-8") as f:
        sentence, pos_tags, ner_tags = [], [], []
        for line in f:
            stripped = line.strip()
            if stripped:
                parts = stripped.split("\t")
                if len(parts) == 3:
                    w, p, n = parts
                    # Strip spaces AND literal quote characters
                    w_clean = w.strip().strip("'\"")
                    
                    if w_clean:
                        sentence.append(w_clean)
                        pos_tags.append(p.strip())
                        ner_tags.append(n.strip())
            elif sentence:
                raw_sentences.append(list(zip(sentence, pos_tags, ner_tags)))
                for i in range(len(sentence)):
                    sparse_features.append(token_to_features(sentence, pos_tags, i))
                    dense_features.append(get_dense_embeddings(sentence, i))
                    labels.append(ner_tags[i])
                sentence, pos_tags, ner_tags = [], [], []
                
        if sentence:
            raw_sentences.append(list(zip(sentence, pos_tags, ner_tags)))
            for i in range(len(sentence)):
                sparse_features.append(token_to_features(sentence, pos_tags, i))
                dense_features.append(get_dense_embeddings(sentence, i))
                labels.append(ner_tags[i])

    return sparse_features, np.array(dense_features), labels, raw_sentences
    
# Load Datasets
train_sparse, train_dense, train_y, train_sents = prepare_dataset("/kaggle/input/datasets/phyohtetpwint/myner-corpus/7-tags/corpus_ver.1.0/train_v5.conll")
test_sparse, test_dense, test_y, test_sents = prepare_dataset("/kaggle/input/datasets/phyohtetpwint/myner-corpus/7-tags/corpus_ver.1.0/test_v5.conll")
val_sparse, val_dense, val_y, val_sents = prepare_dataset("/kaggle/input/datasets/phyohtetpwint/myner-corpus/7-tags/corpus_ver.1.0/val_v5.conll")

# Vectorize Sparse Features
vec = DictVectorizer(sparse=True)
train_sparse_x = vec.fit_transform(train_sparse)
test_sparse_x = vec.transform(test_sparse)
val_sparse_x = vec.transform(val_sparse)

# Combine Matrix Representations
train_X = hstack([train_sparse_x, train_dense])
test_X = hstack([test_sparse_x, test_dense])
val_X = hstack([val_sparse_x, val_dense])

# 5. Model Training
start_time = time.time()
print("Training SVM model for NER...")
clf = LinearSVC(C=1.0, max_iter=2000, random_state=42)
clf.fit(train_X, train_y)
print(f"Training completed in {time.time() - start_time:.2f} seconds.")

# Model Validation
val_preds = clf.predict(val_X)
val_f1 = f1_score(val_y, val_preds, average='weighted')
print(f"NER Validation Weighted F1-score: {val_f1:.4f}")

# 6. Evaluation
preds = clf.predict(test_X)
print("\nLinearSVC SVM model with fasttext embeddings for NER tags Classification Report:")
print(classification_report(test_y, preds, digits=4, zero_division=0))

# 7. Print 5 Predicted Sentences from Test Set
print("\n=== 5 Predicted Sentences from Test Set ===")
pred_idx = 0
for i in range(min(5, len(test_sents))):
    sentence_tokens = test_sents[i]
    print(f"\nSentence {i + 1}:")
    print(f"{'Token':<18} {'POS Tag':<12} {'True NER':<12} {'Predicted NER':<12}")
    print("-" * 56)
    for word, pos_tag, true_ner in sentence_tokens:
        pred_ner = preds[pred_idx]
        print(f"{word:<18} {pos_tag:<12} {true_ner:<12} {pred_ner:<12}")
        pred_idx += 1

=== Environment Details ===
Python Version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux 6.12.90+
scikit-learn Version: 1.6.1
Gensim Version: 4.4.0
Training SVM model for NER...
Training completed in 1151.13 seconds.
NER Validation Weighted F1-score: 0.9804

LinearSVC SVM model with fasttext embeddings for NER tags Classification Report:
              precision    recall  f1-score   support

      B-DATE     0.8060    0.8182    0.8120        66
       B-LOC     0.9787    0.9721    0.9754      1182
       B-NUM     0.5000    0.4000    0.4444        15
       B-ORG     0.7500    0.6250    0.6818        48
       B-PER     0.8611    0.9118    0.8857        34
      B-TIME     0.8750    0.7778    0.8235         9
      E-DATE     0.8644    0.7727    0.8160        66
       E-LOC     0.9763    0.9738    0.9750      1182
       E-NUM     0.4167    0.3333    0.3704        15
       E-ORG     0.7317    0.6250    0.6742        48
       E-PER     0.8421    0.9412    0.8889

<span style="font-size: 18px;">***Model 3: Multi-Task Joint Model (Sparse Features Only) — Jointly predicts a merged label space ($Y_{\text{POS\_NER}} = Y_{\text{POS}} \times Y_{\text{NER}}$) without ground-truth POS inputs.***</span>

In [1]:
# Multi-Task Joint (POS + NER) LinearSVC Model (Sparse Features Only)
import numpy as np
import time
import sys
import platform
import sklearn  
from sklearn.feature_extraction import DictVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, f1_score

def check_environment():
    print("=== Environment Details ===")
    print(f"Python Version: {sys.version}")
    print(f"Platform: {platform.system()} {platform.release()}")
    print(f"scikit-learn Version: {sklearn.__version__}")
    print("===========================")

check_environment()

# 1. Feature Extraction (POS features REMOVED to prevent leakage)
def is_numeric(word):
    numeric_chars = set("၁၂၃၄၅၆၇၈၉၀")
    return word.isdigit() or all(char in numeric_chars for char in word)
    
def token_to_multitask_features(sent, i):
    word = sent[i]
    
    features = {
        'bias': 1.0,
        'word': word,
        'prefix_2': word[:2],
        'prefix_3': word[:3],
        'suffix_2': word[-2:],
        'suffix_3': word[-3:],
        'has_hyphen': '-' in word,
        'is_numeric': is_numeric(word),
        'is_latin': word.isascii(),
    }
    
    # Context Windows (Words only)
    if i > 0:
        features['w-1'] = sent[i-1]
    else:
        features['BOS'] = True

    if i > 1:
        features['w-2'] = sent[i-2]

    if i < len(sent) - 1:
        features['w+1'] = sent[i+1]
    else:
        features['EOS'] = True

    if i < len(sent) - 2:
        features['w+2'] = sent[i+2]
        
    return features

# 2. Process Dataset into Joint Target Labels (POS + "+" + NER)
def prepare_multitask_dataset(file_path):
    sparse_features = []
    pos_labels, ner_labels, joint_labels = [], [], []
    raw_sentences = []
    
    with open(file_path, "r", encoding="utf-8") as f:
        sentence, pos_tags, ner_tags = [], [], []
        for line in f:
            stripped = line.strip()
            if stripped:
                parts = stripped.split("\t")
                if len(parts) == 3:
                    w, p, n = parts
                    w_clean = w.strip().strip("'\"")
                    if w_clean:
                        sentence.append(w_clean)
                        pos_tags.append(p.strip())
                        ner_tags.append(n.strip())
            elif sentence:
                raw_sentences.append(list(zip(sentence, pos_tags, ner_tags)))
                for i in range(len(sentence)):
                    sparse_features.append(token_to_multitask_features(sentence, i))
                    p_tag, n_tag = pos_tags[i], ner_tags[i]
                    pos_labels.append(p_tag)
                    ner_labels.append(n_tag)
                    joint_labels.append(f"{p_tag}+{n_tag}") # Joint Target
                sentence, pos_tags, ner_tags = [], [], []
                
        if sentence:
            raw_sentences.append(list(zip(sentence, pos_tags, ner_tags)))
            for i in range(len(sentence)):
                sparse_features.append(token_to_multitask_features(sentence, i))
                p_tag, n_tag = pos_tags[i], ner_tags[i]
                pos_labels.append(p_tag)
                ner_labels.append(n_tag)
                joint_labels.append(f"{p_tag}+{n_tag}")

    return sparse_features, pos_labels, ner_labels, joint_labels, raw_sentences

# Load Data
train_X_raw, train_pos, train_ner, train_joint, train_sents = prepare_multitask_dataset("/kaggle/input/datasets/phyohtetpwint/myner-corpus/7-tags/corpus_ver.1.0/train_v5.conll")
test_X_raw, test_pos, test_ner, test_joint, test_sents = prepare_multitask_dataset("/kaggle/input/datasets/phyohtetpwint/myner-corpus/7-tags/corpus_ver.1.0/test_v5.conll")
val_X_raw, val_pos, val_ner, val_joint, val_sents = prepare_multitask_dataset("/kaggle/input/datasets/phyohtetpwint/myner-corpus/7-tags/corpus_ver.1.0/val_v5.conll")

# Vectorize
vec = DictVectorizer(sparse=True)
train_X = vec.fit_transform(train_X_raw)
test_X = vec.transform(test_X_raw)
val_X = vec.transform(val_X_raw)

# 3. Model Training on Joint Labels
start_time = time.time()
print("Training Multi-Task Joint LinearSVC model (Sparse Features)...")
clf = LinearSVC(C=1.0, max_iter=2000, random_state=42)
clf.fit(train_X, train_joint)
print(f"Training completed in {time.time() - start_time:.2f} seconds.")

# --- ADDED: Model Validation Phase ---
val_joint_preds = clf.predict(val_X)
val_joint_f1 = f1_score(val_joint, val_joint_preds, average='weighted')

# Decode validation NER predictions
val_pred_ner = [pred.rsplit("+", 1)[1] if "+" in pred else "O" for pred in val_joint_preds]
val_ner_f1 = f1_score(val_ner, val_pred_ner, average='weighted')

print(f"Validation Joint (POS+NER) Weighted F1-score: {val_joint_f1:.4f}")
print(f"Validation Decoupled NER Weighted F1-score: {val_ner_f1:.4f}")
# -------------------------------------

# 4. Joint Prediction Decoding Phase (Test Set)
joint_preds = clf.predict(test_X)

pred_pos = []
pred_ner = []
for pred in joint_preds:
    if "+" in pred:
        p, n = pred.rsplit("+", 1)
        pred_pos.append(p)
        pred_ner.append(n)
    else:
        # Fallback safety in case of non-standard label string
        pred_pos.append("unk")
        pred_ner.append("O")

# 5. Evaluate Tasks Separately
print("\n" + "="*50)
print("MULTI-TASK EVALUATION: TASK 1 - POS TAGGING")
print("="*50)
print(classification_report(test_pos, pred_pos, digits=4, zero_division=0))

print("\n" + "="*50)
print("MULTI-TASK EVALUATION: TASK 2 - NER TAGGING")
print("="*50)
print(classification_report(test_ner, pred_ner, digits=4, zero_division=0))

# 6. Sample Output Verification
print("\n=== 5 Predicted Sentences from Test Set (Joint Predictions) ===")
pred_idx = 0
for i in range(min(5, len(test_sents))):
    sentence_tokens = test_sents[i]
    print(f"\nSentence {i + 1}:")
    print(f"{'Token':<18} {'True POS':<10} {'Pred POS':<10} {'True NER':<10} {'Pred NER':<10}")
    print("-" * 62)
    for word, true_p, true_n in sentence_tokens:
        p_p = pred_pos[pred_idx]
        p_n = pred_ner[pred_idx]
        print(f"{word:<18} {true_p:<10} {p_p:<10} {true_n:<10} {p_n:<10}")
        pred_idx += 1

=== Environment Details ===
Python Version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux 6.12.90+
scikit-learn Version: 1.6.1
Training Multi-Task Joint LinearSVC model (Sparse Features)...
Training completed in 538.39 seconds.
Validation Joint (POS+NER) Weighted F1-score: 0.9560
Validation Decoupled NER Weighted F1-score: 0.9784

MULTI-TASK EVALUATION: TASK 1 - POS TAGGING
              precision    recall  f1-score   support

         abb     1.0000    1.0000    1.0000        18
         adj     0.8770    0.8770    0.8770       569
         adv     0.9434    0.8427    0.8902       356
        conj     0.9322    0.9486    0.9403       739
          fw     0.9552    0.9846    0.9697        65
         int     0.8824    0.8824    0.8824        17
           n     0.9814    0.9821    0.9817      7694
         num     0.9984    0.9828    0.9906       641
        part     0.9765    0.9789    0.9777      4461
         ppm     0.9901    0.9932    0.9916      4114
       

<span style="font-size: 18px;">***Model 4: Multi-Task Joint Model (Sparse + FastText Embeddings) — Jointly predicts POS and NER tags simultaneously using a combined representation of sparse features and FastText dense vectors.***</span>

In [1]:
# Multi-Task Joint (POS + NER) LinearSVC Model (Dense FastText + Sparse Features)
import numpy as np
import time
import sys
import platform
import sklearn  
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction import DictVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, f1_score
import fasttext
import fasttext.util

def check_environment():
    print("=== Environment Details ===")
    print(f"Python Version: {sys.version}")
    print(f"Platform: {platform.system()} {platform.release()}")
    print(f"scikit-learn Version: {sklearn.__version__}")
    print("===========================")

check_environment()

# 1. Load Pre-trained FastText Model for Burmese
print("Loading FastText Burmese model...")
# Path to downloaded FastText model file (e.g., cc.my.300.bin)
ft_model_path = "/kaggle/input/datasets/phyohtetpwint/fast-text/cc.my.300.bin" 
ft = fasttext.load_model(ft_model_path)
EMBEDDING_DIM = ft.get_dimension()
print(f"FastText model loaded successfully. Embedding dimension: {EMBEDDING_DIM}")

# Helper Functions
def is_numeric(word):
    numeric_chars = set("၁၂၃၄၅၆၇၈၉၀")
    return word.isdigit() or all(char in numeric_chars for char in word)

def token_to_multitask_features(sent, i):
    word = sent[i]
    
    features = {
        'bias': 1.0,
        'word': word,
        'prefix_2': word[:2],
        'prefix_3': word[:3],
        'suffix_2': word[-2:],
        'suffix_3': word[-3:],
        'has_hyphen': '-' in word,
        'is_numeric': is_numeric(word),
        'is_latin': word.isascii(),
    }
    
    # Context Windows (Words only - NO POS tags to prevent leakage)
    if i > 0:
        features['w-1'] = sent[i-1]
    else:
        features['BOS'] = True

    if i > 1:
        features['w-2'] = sent[i-2]

    if i < len(sent) - 1:
        features['w+1'] = sent[i+1]
    else:
        features['EOS'] = True

    if i < len(sent) - 2:
        features['w+2'] = sent[i+2]
        
    return features

# 2. Process Dataset into Joint Target Labels & Extract Dense Vectors
def prepare_multitask_dense_dataset(file_path):
    sparse_features = []
    dense_embeddings = []
    pos_labels, ner_labels, joint_labels = [], [], []
    raw_sentences = []
    
    with open(file_path, "r", encoding="utf-8") as f:
        sentence, pos_tags, ner_tags = [], [], []
        for line in f:
            stripped = line.strip()
            if stripped:
                parts = stripped.split("\t")
                if len(parts) == 3:
                    w, p, n = parts
                    w_clean = w.strip().strip("'\"")
                    if w_clean:
                        sentence.append(w_clean)
                        pos_tags.append(p.strip())
                        ner_tags.append(n.strip())
            elif sentence:
                raw_sentences.append(list(zip(sentence, pos_tags, ner_tags)))
                for i in range(len(sentence)):
                    w_clean = sentence[i]
                    p_tag, n_tag = pos_tags[i], ner_tags[i]
                    
                    # Feature & Label collection
                    sparse_features.append(token_to_multitask_features(sentence, i))
                    pos_labels.append(p_tag)
                    ner_labels.append(n_tag)
                    joint_labels.append(f"{p_tag}+{n_tag}") # Composite Joint Target
                    
                    # FastText Vector Retrieval (Handles OOV via subwords automatically)
                    dense_embeddings.append(ft.get_word_vector(w_clean))
                    
                sentence, pos_tags, ner_tags = [], [], []
                
        if sentence:
            raw_sentences.append(list(zip(sentence, pos_tags, ner_tags)))
            for i in range(len(sentence)):
                w_clean = sentence[i]
                p_tag, n_tag = pos_tags[i], ner_tags[i]
                
                sparse_features.append(token_to_multitask_features(sentence, i))
                pos_labels.append(p_tag)
                ner_labels.append(n_tag)
                joint_labels.append(f"{p_tag}+{n_tag}")
                dense_embeddings.append(ft.get_word_vector(w_clean))

    return sparse_features, np.array(dense_embeddings), pos_labels, ner_labels, joint_labels, raw_sentences

# Load Data
print("\nProcessing Train Set...")
train_sparse, train_dense, train_pos, train_ner, train_joint, train_sents = prepare_multitask_dense_dataset("/kaggle/input/datasets/phyohtetpwint/myner-corpus/7-tags/corpus_ver.1.0/train_v5.conll")

print("Processing Test Set...")
test_sparse, test_dense, test_pos, test_ner, test_joint, test_sents = prepare_multitask_dense_dataset("/kaggle/input/datasets/phyohtetpwint/myner-corpus/7-tags/corpus_ver.1.0/test_v5.conll")

print("Processing Validation Set...")
val_sparse, val_dense, val_pos, val_ner, val_joint, val_sents = prepare_multitask_dense_dataset("/kaggle/input/datasets/phyohtetpwint/myner-corpus/7-tags/corpus_ver.1.0/val_v5.conll")

# Vectorize Sparse Features and Concatenate with FastText Dense Vectors
print("\nVectorizing Sparse Features and Stacking FastText Embeddings...")
vec = DictVectorizer(sparse=True)
train_X_sparse = vec.fit_transform(train_sparse)
test_X_sparse = vec.transform(test_sparse)
val_X_sparse = vec.transform(val_sparse)

# Combine Sparse Matrix + Dense FastText Array into a Unified Sparse Matrix
train_X = hstack([train_X_sparse, csr_matrix(train_dense)])
test_X = hstack([test_X_sparse, csr_matrix(test_dense)])
val_X = hstack([val_X_sparse, csr_matrix(val_dense)])

print(f"Combined Feature Matrix Shape (Train): {train_X.shape}")

# 3. Model Training on Joint Labels
start_time = time.time()
print("\nTraining Multi-Task Joint LinearSVC model (Sparse + FastText Features)...")
clf = LinearSVC(
    C=1.0, 
    dual=False,         
    tol=1e-3,           
    max_iter=1000, 
    random_state=42
)

clf.fit(train_X, train_joint)
print(f"Training completed in {time.time() - start_time:.2f} seconds.")

# 4. Model Validation Phase
print("\n--- Model Validation Phase ---")
val_joint_preds = clf.predict(val_X)
val_joint_f1 = f1_score(val_joint, val_joint_preds, average='weighted')

# Decode validation predictions into NER tags for validation tracking
val_pred_ner = [pred.rsplit("+", 1)[1] if "+" in pred else "O" for pred in val_joint_preds]
val_ner_f1 = f1_score(val_ner, val_pred_ner, average='weighted')

print(f"Validation Joint (POS+NER) Weighted F1-score: {val_joint_f1:.4f}")
print(f"Validation Decoupled NER Weighted F1-score: {val_ner_f1:.4f}")

# 5. Joint Prediction Decoding Phase (Test Set)
joint_preds = clf.predict(test_X)

pred_pos = []
pred_ner = []
for pred in joint_preds:
    if "+" in pred:
        p, n = pred.rsplit("+", 1)
        pred_pos.append(p)
        pred_ner.append(n)
    else:
        pred_pos.append("unk")
        pred_ner.append("O")

# 6. Evaluate Tasks Separately
print("\n" + "="*50)
print("MULTI-TASK EVALUATION: TASK 1 - POS TAGGING")
print("="*50)
print(classification_report(test_pos, pred_pos, digits=4, zero_division=0))

print("\n" + "="*50)
print("MULTI-TASK EVALUATION: TASK 2 - NER TAGGING")
print("="*50)
print(classification_report(test_ner, pred_ner, digits=4, zero_division=0))

# 7. Sample Output Verification
print("\n=== 5 Predicted Sentences from Test Set (Joint Predictions) ===")
pred_idx = 0
for i in range(min(5, len(test_sents))):
    sentence_tokens = test_sents[i]
    print(f"\nSentence {i + 1}:")
    print(f"{'Token':<18} {'True POS':<10} {'Pred POS':<10} {'True NER':<10} {'Pred NER':<10}")
    print("-" * 62)
    for word, true_p, true_n in sentence_tokens:
        p_p = pred_pos[pred_idx]
        p_n = pred_ner[pred_idx]
        print(f"{word:<18} {true_p:<10} {p_p:<10} {true_n:<10} {p_n:<10}")
        pred_idx += 1

=== Environment Details ===
Python Version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux 6.12.90+
scikit-learn Version: 1.6.1
Loading FastText Burmese model...
FastText model loaded successfully. Embedding dimension: 300

Processing Train Set...
Processing Test Set...
Processing Validation Set...

Vectorizing Sparse Features and Stacking FastText Embeddings...
Combined Feature Matrix Shape (Train): (201769, 87757)

Training Multi-Task Joint LinearSVC model (Sparse + FastText Features)...
Training completed in 2278.34 seconds.

--- Model Validation Phase ---
Validation Joint (POS+NER) Weighted F1-score: 0.9572
Validation Decoupled NER Weighted F1-score: 0.9790

MULTI-TASK EVALUATION: TASK 1 - POS TAGGING
              precision    recall  f1-score   support

         abb     1.0000    1.0000    1.0000        18
         adj     0.8856    0.8840    0.8848       569
         adv     0.9379    0.8483    0.8909       356
        conj     0.9360    0.9499    0.9429     

# Comparison

<span style="font-size: 18px;">**1. Model 1: Single-Task (Sparse + Gold POS Input)**</span>

<span style="font-size: 18px;">    * Pros: Fastest training time (129.16s); strong Baseline Macro F1 (0.7080).</span>

<span style="font-size: 18px;">    * Cons: Feature Leakage: Relying on gold POS tags makes it non-functional for raw text deployment.</span>

<span style="font-size: 18px;">**2. Model 2: Single-Task (Sparse + FastText + Gold POS Input)**</span>

<span style="font-size: 18px;">    * Pros: Achieves the highest overall NER performance across the board (0.9822 Weighted F1, 0.7096 Macro F1); significantly improves rare entity boundary detection (B-ORG F1 jumps from 0.643 to 0.682).</span>

<span style="font-size: 18px;">    * Cons: Still suffers from feature leakage due to gold POS reliance; training time increases to 1,151.13s due to dense feature processing.</span>

<span style="font-size: 18px;">**3. Model 3: Multi-Task Joint (Sparse Features Only)**</span>

<span style="font-size: 18px;">    * Pros: Realistic end-to-end model needing no gold POS inputs; reasonable training time (538.39s).</span>

<span style="font-size: 18px;">    * Cons: Composite label space ($Y_{\text{POS\_NER}}$) causes target label fragmentation, dropping Macro F1 to 0.6629.</span>

<span style="font-size: 18px;">**4. Model 4: Multi-Task Joint (Sparse + FastText Embeddings)**</span>

<span style="font-size: 18px;">    * Pros: Best end-to-end production setup; achieves 97.76% POS Accuracy and 0.9814 NER Weighted F1 from raw text; subwords rescue OOV entities (B-PER recovers to 0.8986).</span>

<span style="font-size: 18px;">    * Cons: Highest training time (2,278.34s) due to continuous dense vectors combined with multi-class composite hyperplanes.</span>

# Conclusion

<span style="font-size: 18px;">**Model 2** yields the best raw NER metrics (0.7096 Macro F1), proving that subword FastText embeddings provide strong semantic cues for complex entities like organizations (B-ORG/E-ORG). 
However, because Models 1 and 2 require ground-truth POS inputs, they cannot be deployed on unannotated text.</span>

<span style="font-size: 18px;">**Model 4** (Multi-Task Joint with FastText Embeddings) is recommended for real-world deployment. It functions directly on raw text, achieves top-tier POS accuracy (97.76%), and matches single-task weighted NER performance (0.9814 F1) without suffering from feature leakage or cascading pipeline errors.</span>